# Mapping between IPC and industrial sectors (`TLS902_IPC_NACE2`)

The `TLS902_IPC_NACE2` is a reference table containing the link between the IPC classification and the NACE2 codes for industrial sectors. In this way the applications are grouped based on the industry. The NACE2 classification is the Statistical Classification of Economic Activities in the European Community. The concordance table between the NACE2 and the IPC on which the `TLS902_IPC_NACE2` is based was provided by EUROSTAT in co-operation with the KU Leven/Belgium.

The `TLS902_IPC_NACE2` table is a reference table that links the IPC classification to NACE2 codes, which represent industrial sectors. Through this linkage, patent applications can be grouped according to their corresponding industries. The NACE2 classification is the Statistical Classification of Economic Activities in the European Community. The concordance table between IPC and NACE2, on which `TLS902_IPC_NACE2` is based, was developed by Eurostat in cooperation with KU Leuven/Belgium. 

Let’s break down the key fields in this table. The first step as usual is to initialise the PATSTAT client and import the table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS902_IPC_NACE2

# Importing sql alchemy package
from sqlalchemy import func

## Key fields in the `TLS902_IPC_NACE2` table
* `IPC` (primary key): 
* `NOT_WITH_IPC`: this attribute indicate the IPC main group which must not co-occur with the IPC in attribute IPC. In the most case this field is empty.
* `UNLESS_WITH_IPC`: this attribute indicate the IPC main group which nullifies the effect of the attribute NOT_WITH_IPC column if it co-occurs with the symbol in the attribute IPC.
* `NACE2_CODE`: NACE2 is the Statistical Classification of Economics Activities in the European Community. NACE2_CODE is the code associated to a defined category of economic activity.
* `NACE2_WEIGHT`: this attribute indicate whether there is a mapping between a particular IPC and a NACE2 code. It can take either value 1 *or* 0. Value 1 indicates that there is a mapping between IPC and NACE2, while value 0 indicates that no mapping exists.
* `NACE2_DESCR`:this attribute is the description of the NACE2 code indicating the industrial sector.


To determine the industry (NACE2 code) corresponding to each application ID, the table ``TLS902_IPC_NACE2`` can be joined with ``TLS209_APPLN_IPC`` using ``ipc_class_symbol`` as the linking key. The attribute `ipc_class_symbol` can have up to 14 characters, whereas `ipc` has only four. For this reason, when merging the two tables, it is important to consider only the first four characters of both attributes (identifing the IPC subclasses) by truncating `ipc_class_symbol`. To do this, we use the `substring` function from SQLAlchemy’s `func` module.

In [3]:
#Import the two tables TLS902_IPC_NACE2 and TLS209_APPLN_IPC 
from epo.tipdata.patstat.database.models import TLS902_IPC_NACE2, TLS209_APPLN_IPC 

#Join the two tables using the IPC attribute in order to retrieve the industrial fields associated with each application id
industrial_id = (
    db.query(
        TLS209_APPLN_IPC.appln_id,
         TLS902_IPC_NACE2.ipc,
        TLS902_IPC_NACE2.nace2_code,
       TLS902_IPC_NACE2.nace2_weight,
        TLS902_IPC_NACE2.nace2_descr,
    )

# NB: The attribute `ipc_class_symbol` can have up to fourteen characters, whereas `ipc` has only four. For this reason, when merging the two tables,it is important to consider only the first 4 characters of both attributes (identifing the IPC subclasses) by truncating `ipc_class_symbol`.
# To do this, we use the `substring` function from SQLAlchemy’s `func` module.

    .join(
        TLS902_IPC_NACE2,
        TLS902_IPC_NACE2.ipc== func.substr( TLS209_APPLN_IPC.ipc_class_symbol,1,4)  
    )
        .order_by(
           TLS209_APPLN_IPC.appln_id #Order by the application id
        )        .distinct()  #Add " distinct" to remove duplicate
    .limit(10000)
)

#Show the results as a dataframe
industrial_df= patstat.df(industrial_id)
industrial_df

,appln_id,ipc,nace2_code,nace2_weight,nace2_descr
0,1,H04W,26.3,1,Manufacture of Communication Equipment
1,1,G06K,28.23,1,Manufacture of Office Machinery and Equipment ...
2,1,H01R,27.33,1,Manufacture of Wiring Devices
3,1,H04M,26.3,1,Manufacture of Communication Equipment
4,2,C07K,21,1,Manufacture of Basic Pharmaceutical Products a...
...,...,...,...,...,...
9995,6841,A61L,32.5,1,Manufacture of medical and dental instruments ...
9996,6841,A61F,32.5,1,Manufacture of medical and dental instruments ...
9997,6841,A61K,21,1,Manufacture of Basic Pharmaceutical Products a...
9998,6843,A61B,32.5,1,Manufacture of medical and dental instruments ...
